In [ ]:
import os

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from rdkit import RDLogger
from rdkit.Chem import AllChem, CanonSmiles, MolFromSmiles, MolToSmiles, PandasTools
from rdkit.Contrib.NP_Score import npscorer
from sklearn.decomposition import PCA
from tqdm import tqdm

from guild.tools.ligand_properties import (
    assign_properties,
    compound_filter,
    generate_equal_mw_distributions,
)

RDLogger.DisableLog("rdApp.info")
lg = RDLogger.logger()
lg.setLevel(RDLogger.CRITICAL)
tqdm.pandas()

In [ ]:
publication_data_folder = "../../guild/support/publication_data"
os.makedirs(publication_data_folder, exist_ok=True)

In [ ]:
MIN_MW = 250
MAX_MW = 450
MAX_RING_SIZE = 10

In [ ]:
def canonicalize_smiles(smiles):
    try:
        return CanonSmiles(smiles)
    except Exception:
        return smiles

# Preparing the NP dataset

In [ ]:
loaded_nps = pd.read_csv(
    "../../guild/support/coconut_csv-02-2026.csv", sep=",", header=0
)[["identifier","canonical_smiles"]]
loaded_nps.columns = ["COCONUT_ID","SMILES"]
loaded_nps = loaded_nps.drop_duplicates(subset="SMILES").reset_index(drop=True)

In [ ]:
loaded_nps[["simplified_id","0_scaffold"]] = loaded_nps["COCONUT_ID"].str.split(".", expand=True)
loaded_nps = loaded_nps.loc[loaded_nps["0_scaffold"] == "0"].reset_index(drop=True)

In [ ]:
loaded_nps["can_smiles"] = loaded_nps["SMILES"].progress_apply(canonicalize_smiles)

loaded_nps = loaded_nps[["COCONUT_ID", "can_smiles"]]
loaded_nps = loaded_nps.reset_index(drop=True)
loaded_nps.head(2)

### Adding chemical properties

In [ ]:
#if not os.path.exists("../../guild/support/coconut_properties.tsv"):
loaded_nps_properties = assign_properties(loaded_nps)

# Replace empty strings with None
loaded_nps_properties["scaffold_smiles"] = loaded_nps_properties[
    "scaffold_smiles"
].replace("", None)

loaded_nps_properties.to_csv(
    "../../guild/support/coconut_properties.tsv", sep="\t", index=None
)
#else:
#    loaded_nps_properties = pd.read_csv(
#        "../../guild/support/coconut_properties.tsv", sep="\t"
#    )

loaded_nps_properties.head(2)

In [ ]:
loaded_nps_with_properties = pd.merge(
    loaded_nps,
    loaded_nps_properties,
    left_on="COCONUT_ID",
    right_on="id",
    how="inner",
)
loaded_nps_with_properties.drop(columns=["id"], inplace=True)

#if not os.path.exists(f"{publication_data_folder}/np_with_properties.tsv"):
loaded_nps_with_properties.to_csv(
    f"{publication_data_folder}/np_with_properties.tsv", sep="\t", index=None
)

len(loaded_nps_with_properties)

### Subsetting to properties within drug-like space

In [ ]:
loaded_nps_with_properties = loaded_nps_with_properties.dropna(
    subset=["molecular_weight", "scaffold_smiles"], how="any"
)

In [ ]:
if not os.path.exists(f"{publication_data_folder}/np_filtered_with_properties.tsv"):
    nps_filtered = compound_filter(
        loaded_nps_with_properties,
        min_MW=MIN_MW,
        max_MW=MAX_MW,
        scaffold_col="scaffold_smiles",
        ro5_fulfilled_col="ro5_fulfilled",
        largest_ring_size_col="max_ring_size",
        max_per_scaffold=5,
        seed=42,
        MW_column="molecular_weight",
        max_ring_size=MAX_RING_SIZE,
    )

    # Add activity column
    nps_filtered["activity"] = "Natural Product"
    nps_filtered.to_csv(
        f"{publication_data_folder}/np_filtered_with_properties.tsv",
        sep="\t",
        index=False,
    )

else:
    nps_filtered = pd.read_csv(
        f"{publication_data_folder}/np_filtered_with_properties.tsv",
        sep="\t",
        header=0,
    )

f"Filtered the NPs from {len(loaded_nps)} to {len(nps_filtered)}"

# Preparing the synthetic dataset

The dataset is from Enamine and cannot be released due to legal issues. Hence, IDs have been masked in the final output.

In [ ]:
loaded_synthetics = PandasTools.LoadSDF(
    "../../guild/support/Enamine_Hit_Locator_Library_plated_200000cmpds_20210816.sdf"
)
loaded_synthetics["smiles"] = loaded_synthetics.ROMol.progress_apply(MolToSmiles)
loaded_synthetics["can_smiles"] = loaded_synthetics["smiles"].progress_apply(
    canonicalize_smiles
)

loaded_synthetics = loaded_synthetics[["Catalog ID", "can_smiles"]]

### Adding chemical properties

In [ ]:
if not os.path.exists("../../guild/support/enamine_properties.tsv"):
    loaded_synthetics_properties = assign_properties(loaded_synthetics)

    # Replace empty strings with None
    loaded_synthetics_properties["scaffold_smiles"] = loaded_synthetics_properties[
        "scaffold_smiles"
    ].replace("", None)

    loaded_synthetics_properties.to_csv(
        "../../guild/support/enamine_properties.tsv", sep="\t", index=None
    )
else:
    loaded_synthetics_properties = pd.read_csv(
        "../../guild/support/enamine_properties.tsv", sep="\t"
    )

loaded_synthetics_properties.head(2)

In [ ]:
loaded_synthetics_with_properties = pd.merge(
    loaded_synthetics,
    loaded_synthetics_properties,
    left_on="Catalog ID",
    right_on="id",
    how="inner",
)
loaded_synthetics_with_properties.drop(columns=["id"], inplace=True)

if not os.path.exists(f"{publication_data_folder}/synthetics_with_properties.tsv"):
    loaded_synthetics_with_properties.to_csv(
        f"{publication_data_folder}/synthetics_with_properties.tsv",
        sep="\t",
        index=None,
    )

len(loaded_synthetics_with_properties)

### Subsetting to properties within drug-like space

In [ ]:
loaded_synthetics_with_properties = loaded_synthetics_with_properties.dropna(
    subset=["molecular_weight", "scaffold_smiles"], how="any"
)

In [ ]:
if not os.path.exists(
    f"{publication_data_folder}/synthetics_filtered_with_properties.tsv"
):
    synthetics_filtered = compound_filter(
        loaded_synthetics_with_properties,
        min_MW=MIN_MW,
        max_MW=MAX_MW,
        scaffold_col="scaffold_smiles",
        ro5_fulfilled_col="ro5_fulfilled",
        largest_ring_size_col="max_ring_size",
        max_per_scaffold=5,
        seed=42,
        MW_column="molecular_weight",
        max_ring_size=MAX_RING_SIZE,
    )

    # Add activity column
    synthetics_filtered["activity"] = "Synthetic"
    synthetics_filtered.to_csv(
        f"{publication_data_folder}/synthetics_filtered_with_properties.tsv",
        sep="\t",
        index=False,
    )

else:
    synthetics_filtered = pd.read_csv(
        f"{publication_data_folder}/synthetics_filtered_with_properties.tsv",
        sep="\t",
        header=0,
    )

f"Filtered the Synthetics from {len(loaded_synthetics)} to {len(synthetics_filtered)}"

# Exploratory analysis of the final sets

In [ ]:
filtered_df = pd.concat(
    [synthetics_filtered.reset_index(drop=True), nps_filtered.reset_index(drop=True)],
    axis=0,
)
filtered_df["ID"] = filtered_df["Catalog ID"].fillna(filtered_df["COCONUT_ID"])
filtered_df.drop(columns=["Catalog ID", "COCONUT_ID"], inplace=True)
filtered_df.head(1)

### Checking the molecular weight distrbution

In [ ]:
fig = px.histogram(
    filtered_df,
    x="molecular_weight",
    nbins=100,
    color="activity",
)
fig.show()

### Check the NP score distrbution

In [ ]:
np_model = npscorer.readNPModel()

In [ ]:
filtered_df["npscore"] = filtered_df["can_smiles"].progress_apply(
    lambda x: npscorer.scoreMol(MolFromSmiles(x), np_model)
)

In [ ]:
fig = px.histogram(filtered_df, x="npscore", nbins=100, color="activity")
fig.show()

To have non-overlapping NP space, a cut-off of 0.05 is used.

In [ ]:
nps_binned = filtered_df[
    (filtered_df["npscore"] > 0.0) & (filtered_df["activity"] == "Natural Product")
]
synthetics_binned = filtered_df[
    (filtered_df["npscore"] < -0.5) & (filtered_df["activity"] == "Synthetic")
]

binned_df = pd.concat(
    [nps_binned.reset_index(drop=True), synthetics_binned.reset_index(drop=True)],
    axis=0,
)
len(binned_df)

In [ ]:
fig = px.histogram(binned_df, x="npscore", nbins=100, color="activity")
fig.show()

### Generating ECFP fingerprint and checking their chemical space

In [ ]:
def get_fingerprints(df, smiles_col):
    df["ROMol"] = [
        MolFromSmiles(x)
        for x in tqdm(df[smiles_col], total=len(df), desc="Generating molecules")
    ]
    radius = 3
    nBits = 1024
    ECFP6 = [
        list(AllChem.GetMorganFingerprintAsBitVect(x, radius=radius, nBits=nBits))
        for x in tqdm(df["ROMol"], total=len(df), desc="Generating fingerprints")
    ]
    return pd.DataFrame(ECFP6, index=df.index)

In [ ]:
fingerprints_df = get_fingerprints(binned_df, "can_smiles")

In [ ]:
pca = PCA(n_components=2)
pca.fit(fingerprints_df)
pca_result = pca.transform(fingerprints_df)
pca_result = pd.DataFrame(pca_result, columns=["PC1", "PC2"])
pca_result["activity"] = binned_df["activity"].tolist()

fig = px.scatter(
    data_frame=pca_result, x="PC1", y="PC2", color="activity", opacity=0.25
)
fig.update_layout(
    title="PCA of ECFP fingerprints",
    xaxis_title="PC1",
    yaxis_title="PC2",
    legend_title="Compounds type",
    height=600,
    width=1000,
)
fig.show()

# Generating the final dataset of 1000 samples each

This dataset requires an equal molecular weight distrbution along with sampling about 1,000 compounds from each.

In [ ]:
matched_np, matched_synthetic = matched_np, matched_synthetic = (
    generate_equal_mw_distributions(
        df_1=nps_binned,
        df_2=synthetics_binned,
        mw_column="molecular_weight",
        lipinski_column="ro5_fulfilled",
        n_bins=10,
        target_size=1000,
    )
)

In [ ]:
len(matched_np), len(matched_synthetic)

In [ ]:
sampled_ligands = pd.concat([matched_np, matched_synthetic], ignore_index=True)
col_order = ["ID"] + [i for i in sampled_ligands.columns.tolist() if i != "ID"]
sampled_ligands = sampled_ligands[col_order]
sampled_ligands.head(2)

In [ ]:
sampled_ligands.to_csv(
    f"{publication_data_folder}/ligands_sampled_1000.tsv",
    sep="\t",
    index=False,
)

### Inspecting property of final set

In [ ]:
# make three histograms, one for each activity, one for each logp, one for each fsp3, one for each topological_surface_area_mapping with subplots
columns = ["logp", "fsp3", "topological_surface_area_mapping", "max_ring_size"]
activities = sampled_ligands["activity"].unique()

fig = make_subplots(rows=4, cols=1, subplot_titles=columns)

for i, col in enumerate(columns, start=1):
    for activity in activities:
        data = sampled_ligands[sampled_ligands["activity"] == activity][col]
        fig.add_trace(
            go.Histogram(
                x=data,
                nbinsx=100,
                name=activity,
                legendgroup=activity,
                showlegend=True,
            ),
            row=i,
            col=1,
        )

fig.update_layout(barmode="overlay", height=800)
fig.update_traces(opacity=0.7)
fig.show()

In [ ]:
subset_fingerprints_df = get_fingerprints(sampled_ligands, "can_smiles")

In [ ]:
pca = PCA(n_components=2)
pca.fit(fingerprints_df)
pca_result = pca.transform(fingerprints_df)
pca_result = pd.DataFrame(pca_result, columns=["PC1", "PC2"])
pca_result["activity"] = binned_df["activity"].tolist()

fig = px.scatter(
    data_frame=pca_result, x="PC1", y="PC2", color="activity", opacity=0.25
)
fig.update_layout(
    title="PCA of ECFP fingerprints Sampled Ligands only",
    xaxis_title="PC1",
    yaxis_title="PC2",
    legend_title="Compounds type",
    height=600,
    width=1000,
)
fig.show()